<a href="https://colab.research.google.com/github/romellfudi/medium/blob/main/vector_db/Function%20Calling%20-%20Metadata%20milvus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain pydantic python-dotenv openai pypdf tiktoken faiss-cpu chromadb lark "milvus[client]" sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.6/276.6 kB 34.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 107.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 120.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.8/479.8 kB 49.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 14.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.9/92.9 kB 11.6 MB/

**Must restart the runtime in order to milvus works correctly**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/_openai/
import os
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.
%cd /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/_openai
/content


In [ ]:
import openai
# openai.api_key

from pydantic import BaseModel, Field

from langchain.chat_models import ChatOpenAI
from langchain.agents import Tool
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA

from transformers import AutoTokenizer, AutoModel


In [ ]:
embeddings = OpenAIEmbeddings()

%cd /content/drive/MyDrive/_openai/
directory_path = "/content/drive/MyDrive/_openai/"
vector_stores = {}

for dir_name in os.listdir(directory_path):
    if dir_name.endswith("_faiss_index"):
        index_path = os.path.join(directory_path, dir_name)
        vector_stores[dir_name.split('_')[0]] = FAISS.load_local(index_path, embeddings)

%cd /content/

/content/drive/MyDrive/_openai
/content


In [ ]:
list_docs = []
[ list_docs.extend(v.similarity_search(" ", k=99)) for v in vector_stores.values() ]
len(list_docs)

10

In [ ]:
for doc in list_docs:
    print(doc.metadata)

{'source': 'https://drive.google.com/file/d/1uouVQfOY9gEtiuzgsSIGwu6zVfOT2APk/view?usp=sharing', 'page': 0, 'full_name': 'Jorge Gaya', 'proffesion_field': 'Private Tutor', 'country': 'Peru', 'genre': 'Male', 'languages': ['Spanish'], 'years_experience': '10', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Communication'], 'hard_skills': ['Planning and Methodology', 'Pedagogical Knowledge', 'Topic Preparation']}
{'source': 'https://drive.google.com/file/d/1s3Dm7FP3z6dvZ6JeIMlIRgcg0eoGf3pk/view?usp=sharing', 'page': 0, 'full_name': 'Raúl Meléndez', 'proffesion_field': 'Software Developer', 'country': 'Peru', 'genre': 'Male', 'languages': ['Spanish', 'English', 'French'], 'years_experience': '3', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Communication', 'Teamwork'], 'hard_skills': ['Database', 'Cache', 'HTML5', 'CSS', 'JavaScript', 'App Architecture', 'Editing Programs']}
{'source': 'https://drive.google.com/file/d/1qWropNuTE_2Zj-WQdX44ULuXDf1c7O1H/view?usp=sharing', 'page': 0, 'fu

In [ ]:
embedding_model="thenlper/gte-base"
hf_tokenizer = AutoTokenizer.from_pretrained(embedding_model)
hf_model = AutoModel.from_pretrained(embedding_model)

def embedding_text(input_text):
    batch_dict = hf_tokenizer([input_text.strip().lower()], max_length=512, padding=True, truncation=True, return_tensors='pt')
    outputs = hf_model(**batch_dict)
    # do a masked mean over the dimension(average_pool)
    last_hidden = outputs.last_hidden_state.masked_fill(~batch_dict['attention_mask'][..., None].bool(), 0.0)
    torch_embeddings_list = last_hidden.sum(dim=1) / batch_dict['attention_mask'].sum(dim=1)[..., None]
    return torch_embeddings_list[0].tolist() # dimension of 768

In [ ]:
print(len(embedding_text("hi")))

768


In [ ]:
#@title Initilize the Milvus server
from milvus import default_server
from pymilvus import connections, utility

default_server.start()
connections.connect(host='127.0.0.1', port=default_server.listen_port)

display(utility.get_server_version())

'v2.3.1-lite'

In [ ]:
#@title Create the schema of the collection
from pymilvus import FieldSchema, CollectionSchema, DataType, Collection
COLLECTION_NAME = "CVs"
DIMENSION = 768
# Check that the collection does not yet exist
if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)

fields = [
    FieldSchema(name='id', dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name='cv_full_text', dtype=DataType.VARCHAR, max_length=4000),
    FieldSchema(name='cv_metadata', dtype=DataType.VARCHAR, max_length=4000),
    FieldSchema(name='embedding_metadata', dtype=DataType.FLOAT_VECTOR, dim=DIMENSION)
]
schema = CollectionSchema(fields=fields)
collection = Collection(name=COLLECTION_NAME, schema=schema)

In [ ]:
index_params = {
    "index_type": "IVF_FLAT",
    "metric_type": "L2",
    "params": {"nlist": 128},
}
collection.create_index(field_name="embedding_metadata", index_params=index_params)
collection.load() # the collection is ready to work

In [ ]:
#@title Let's ingest the data
from pprint import pprint

def ingest_cv_documents(list_of_documents, milvus_collection):
    # A list with entities
    list_of_entities = [
                [], # list of cv's full text
                [], # list of cv's metadata
                [] # list of embeddings of cvs's metadata
    ]
    for document in list_of_documents:
        embeddings = embedding_text(str(document.metadata)) # A numerical vector
        list_of_entities[0].append(document.page_content)
        list_of_entities[1].append(str(document.metadata))
        list_of_entities[2].append(embeddings)

    # pprint(list_of_entities)
    ids = milvus_collection.insert(list_of_entities)
    if ids:
        milvus_collection.flush()
        print(f"{ids.succ_count} entities were inserted")
    else:
        print("No entity was inserted.")

ingest_cv_documents(list_docs, collection)

10 entities were inserted


In [ ]:
#@title Searching for Text Related to the Input Inquiry
embeds = embedding_text("who candidates has female as genre ") # return an embedding
TOP_K = 4
result = collection.search([embeds], # Wrap the embedding in an inquiry list
                  anns_field='embedding_metadata',
                  param={"metric_type": "L2",
                        "params": {"nprobe": 10}},
                  limit=TOP_K, # Number of most similar embeddings
                  output_fields=['cv_metadata']) # Data associated with vectors

In [ ]:
#@title Displaying the results
for hits_i, hits in enumerate(result):
    for hit_it, hit in enumerate(hits):
        pprint(str(hit.entity), width=150)

('id: 445166726216679591, distance: 108.84059143066406, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-educacion-da4453.pdf\', '
 "'page': 0, 'full_name': 'Samanta Torres', 'proffesion_field': 'Education', 'country': 'Lima', 'genre': 'Female', 'languages': ['Spanish', "
 "'English', 'German'], 'years_experience': '4', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Patience', 'Persuasion', 'Teamwork'], "
 '\'hard_skills\': [\'Communication Skills\']}"}')
('id: 445166726216679592, distance: 110.0274658203125, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-gerente-sucursal.pdf\', '
 "'page': 0, 'full_name': 'Ana López', 'proffesion_field': 'Branch Manager', 'country': 'Lima', 'genre': 'Female', 'languages': [], "
 "'years_experience': 4, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Customer Service', 'Stress Management', 'Leadership'], 'hard_skills': "
 '[\'Organization\', \'Financial Skills\', \'Marketing\']}"}')
('id: 445166726216679593, distance: 110.4

In [ ]:
import json
def search_candidates_by(user_intention, k = 2, retrieve_data='cv_metadata'):
    embeds = embedding_text(user_intention)
    result = collection.search([embeds],
                      anns_field='embedding_metadata',
                      param={"metric_type": "L2",
                            "params": {"nprobe": 10}},
                      limit=k,
                      output_fields=[retrieve_data])
    results = []
    for hits_i, hits in enumerate(result):
      for hit_it, hit in enumerate(hits):
          results.append(hit.entity.get(retrieve_data))
          pprint(str(hit.entity), width=150)

    return json.dumps(results)

available_functions = {
    'search_candidates_by': search_candidates_by
}

In [ ]:
print(search_candidates_by('{"genre":"female"}',k=1))

('id: 445166726216679592, distance: 108.88485717773438, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-gerente-sucursal.pdf\', '
 "'page': 0, 'full_name': 'Ana López', 'proffesion_field': 'Branch Manager', 'country': 'Lima', 'genre': 'Female', 'languages': [], "
 "'years_experience': 4, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Customer Service', 'Stress Management', 'Leadership'], 'hard_skills': "
 '[\'Organization\', \'Financial Skills\', \'Marketing\']}"}')
["{'source': '/content/cv-ejemplo-gerente-sucursal.pdf', 'page': 0, 'full_name': 'Ana L\u00f3pez', 'proffesion_field': 'Branch Manager', 'country': 'Lima', 'genre': 'Female', 'languages': [], 'years_experience': 4, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Customer Service', 'Stress Management', 'Leadership'], 'hard_skills': ['Organization', 'Financial Skills', 'Marketing']}"]


In [ ]:
print(search_candidates_by('{"soft_skills":"communcation","hard_skills":"software"}',k=2, retrieve_data='cv_metadata'))

('id: 445166726216679592, distance: 68.15179443359375, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-gerente-sucursal.pdf\', '
 "'page': 0, 'full_name': 'Ana López', 'proffesion_field': 'Branch Manager', 'country': 'Lima', 'genre': 'Female', 'languages': [], "
 "'years_experience': 4, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Customer Service', 'Stress Management', 'Leadership'], 'hard_skills': "
 '[\'Organization\', \'Financial Skills\', \'Marketing\']}"}')
('id: 445166726216679595, distance: 68.97846221923828, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-programador-b0292a.pdf\', '
 "'page': 0, 'full_name': 'Raúl Meléndez', 'proffesion_field': 'Software Developer', 'country': 'Peru', 'genre': 'Male', 'languages': ['Spanish', "
 "'English', 'French'], 'years_experience': '3', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Communication', 'Teamwork'], 'hard_skills': "
 '[\'Database\', \'Cache\', \'HTML5\', \'CSS\', \'JavaScript\', \'App Architect

In [ ]:
functions = [
        {
            "name": "search_candidates_by",
            "description": "Get a list of candidates which match better with the user's intent",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_intention": {
                        "type": "string",
                        "description": "A JSON string which contains the keys (full_name, proffesion_field, country, genre, languages, years_experience, email, soft_skills, hard_skills) with values, in case the user doesn't mention",
                    },
                    "k": {
                        "type": "integer",
                        "description": "The number of candidates the user is looking for is set to 3 by default if the user requests multiple candidates. If the user only requests one candidate or uses the singular form, it is set to 1 by default."
                    }
                },
                "required": ["user_intention", "k"],
            },
        }
    ]

In [ ]:
messages = [{'role': 'system', 'content': 'You are a system which make a JSON using the user input to interpretate it into a JSON, you must generate a '}]
user_message="Who candidates are female and are from computer science field?"
messages.append({"role": "user", "content": user_message})
completion= openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions
)

In [ ]:
response_function_calling=completion.choices[0].message
response_function_calling

<OpenAIObject at 0x7bda615d8ef0> JSON: {
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "search_candidates_by",
    "arguments": "{\n  \"user_intention\": \"{\\\"genre\\\": \\\"female\\\", \\\"proffesion_field\\\": \\\"computer science\\\"}\",\n  \"k\": 3\n}"
  }
}

In [ ]:
user_message="Who candidate is female and is from computer science field?"
messages.append({"role": "user", "content": user_message})
response_function_calling = openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions
).choices[0].message
response_function_calling

<OpenAIObject at 0x7bd99eb79030> JSON: {
  "role": "assistant",
  "content": null,
  "function_call": {
    "name": "search_candidates_by",
    "arguments": "{\n  \"user_intention\": \"{\\\"genre\\\":\\\"female\\\", \\\"proffesion_field\\\":\\\"computer science\\\"}\",\n  \"k\": 1\n}"
  }
}

In [ ]:
function_name=(response_function_calling['function_call'])['name']
print(function_name)
import json
parameters=eval(response_function_calling['function_call']['arguments'])
print(parameters)

search_candidates_by
{'user_intention': '{"genre":"female", "proffesion_field":"computer science"}', 'k': 1}


In [ ]:
print(available_functions[function_name]( **parameters, ))

('id: 445166726216679591, distance: 64.21623229980469, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-educacion-da4453.pdf\', '
 "'page': 0, 'full_name': 'Samanta Torres', 'proffesion_field': 'Education', 'country': 'Lima', 'genre': 'Female', 'languages': ['Spanish', "
 "'English', 'German'], 'years_experience': '4', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Patience', 'Persuasion', 'Teamwork'], "
 '\'hard_skills\': [\'Communication Skills\']}"}')
["{'source': '/content/cv-ejemplo-educacion-da4453.pdf', 'page': 0, 'full_name': 'Samanta Torres', 'proffesion_field': 'Education', 'country': 'Lima', 'genre': 'Female', 'languages': ['Spanish', 'English', 'German'], 'years_experience': '4', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Patience', 'Persuasion', 'Teamwork'], 'hard_skills': ['Communication Skills']}"]


In [ ]:
print(available_functions[function_name]( **parameters, retrieve_data='cv_full_text' ))

("id: 445166726216679591, distance: 64.21623229980469, entity: {'cv_full_text': 'IDIOMAS\\nEspañol\\nInglés\\nAlemán\\nPASATIEMPOS\\nEn mi tiempo "
 'libre me gusta \\nescribir pequeños poemasSAMANTA TORRES\\n¡Recién graduada y muy motivada para contribuir al futuro! Después de 3 prácticas '
 'exitosas en educación\\nprimaria con clases de un promedio de 25 alumnos, estoy convencida de que con mi paciencia y persuasión\\npuedo causar una '
 'impresión positiva en mis alumnos. Disfruto trabajando con otros colegas y disfruto\\npreparando lecciones de forma '
 'independiente.\\nEDUCACIÓN\\nLicenciatura en educación\\nUniversidad de Lima, Lima\\nDiploma de Educación Primaria.\\nCertificado de Habilidades '
 'de Comunicación\\nInstituto ACP, Lima\\nSeguí voluntariamente este curso para fortalecer mis habilidades '
 'de\\ncomunicación.\\nPreparatoria\\nInstituto Peruano, Lima\\nEXPERIENCIA LABORAL Y PRÁCTICAS PROFESIONALES\\nPasantía de graduación\\nEscuela '
 'primaria Pacelli, Lima\\nPasa

In [ ]:
def ask_to_candidates_bot(user_input):
    messages = [{'role': 'system', 'content': 'You are a system which make a JSON using the user input to interpretate it into a JSON, you must generate a '}]
    messages.append({"role": "user", "content": user_input})
    completion = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=messages,
        functions=functions
    )
    response_function_calling = completion.choices[0].message
    if response_function_calling is not None and response_function_calling.get('function_call'):
        # If a function call is detected
        function_name = response_function_calling['function_call']['name']
        function_args = json.loads(response_function_calling["function_call"]["arguments"])
        function_response = available_functions[function_name](**function_args)

        print(f'{"-"*70}\narguments:\n{function_args}\n{"-"*70}\n')

        # Call GPT-3 again with function response
        _response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "user", "content": user_input},
                response_function_calling,
                {
                    "role": "function",
                    "name": function_name,
                    "content": function_response,
                },
            ],
            functions=functions
        )

        return _response.choices[0]["message"]["content"].strip()

    else:
        # If no function call detected, return the initial completion
        return completion.choices[0]["message"]["content"].strip()


In [ ]:
pprint(ask_to_candidates_bot("Who candidate is male and has knowdlege about laws and regulations?"), width=150)

('id: 445166726216679587, distance: 66.7119369506836, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-abogado-406692.pdf\', \'page\': '
 "0, 'full_name': 'Patricio Ugarte', 'proffesion_field': 'Law', 'country': 'Peru', 'genre': 'Male', 'languages': ['Spanish'], 'years_experience': "
 "14, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Clear communication', 'Idealism', 'Morality'], 'hard_skills': ['Labor law', 'Legal research', "
 '\'Litigation\']}"}')
----------------------------------------------------------------------
arguments:
{'user_intention': '{"genre":"male", "hard_skills":"laws and regulations"}', 'k': 1}
----------------------------------------------------------------------

('The candidate that matches your criteria is Patricio Ugarte. Here are some details about him:\n'
 '\n'
 '- Full Name: Patricio Ugarte\n'
 '- Profession Field: Law\n'
 '- Country: Peru\n'
 '- Genre: Male\n'
 '- Languages: Spanish\n'
 '- Years of Experience: 14\n'
 '- Email: ejemplo@cvmake

In [ ]:
print(ask_to_candidates_bot("Who candidates are female and have knowdlege about computer science?"))

('id: 445166726216679591, distance: 64.21623229980469, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-educacion-da4453.pdf\', '
 "'page': 0, 'full_name': 'Samanta Torres', 'proffesion_field': 'Education', 'country': 'Lima', 'genre': 'Female', 'languages': ['Spanish', "
 "'English', 'German'], 'years_experience': '4', 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Patience', 'Persuasion', 'Teamwork'], "
 '\'hard_skills\': [\'Communication Skills\']}"}')
('id: 445166726216679592, distance: 64.57183837890625, entity: {\'cv_metadata\': "{\'source\': \'/content/cv-ejemplo-gerente-sucursal.pdf\', '
 "'page': 0, 'full_name': 'Ana López', 'proffesion_field': 'Branch Manager', 'country': 'Lima', 'genre': 'Female', 'languages': [], "
 "'years_experience': 4, 'email': 'ejemplo@cvmaker.pe', 'soft_skills': ['Customer Service', 'Stress Management', 'Leadership'], 'hard_skills': "
 '[\'Organization\', \'Financial Skills\', \'Marketing\']}"}')
('id: 445166726216679590, distance: 66.306